# CropGuard AI: YOLO11s Agricultural Pest Detection Model Training Workflow

**Project:** CropGuard AI — Intelligent Pest Detection & Crop Protection (B.Tech CSE Capstone)  
**Model Architecture:** Ultralytics YOLO11s (`yolo11s.pt`, ~9.4M parameters)  
**Target Classes (6):**
0. `Aphids`
1. `Whitefly`
2. `Brown Planthopper`
3. `Yellow Stem Borer`
4. `Thrips`
5. `Helicoverpa armigera`

**Hardware Environment:** Google Colab GPU (NVIDIA Tesla T4 16GB VRAM)  
**Key Protocols:**
- **Zero Hardcoded Secrets:** Roboflow API key is securely entered via Colab Secrets (`userdata`) or `getpass`.
- **Strict Anti-Leakage:** SHA-256 deduplication and Roboflow parent-hash grouping ensure no augmented variants cross splits.
- **Class Remapping & Biological Integrity:** Unified `0..5` schema mapping from 3 verified public open-source datasets.
- **Reproducibility:** Deterministic seed `42`, 70% Train / 20% Val / 10% Test split.
- **CUDA OOM Fallback:** Automatic batch size fallback (`16` -> `8`).

## 1. Environment Verification & Dependency Setup

In [ ]:
# Verify GPU acceleration
!nvidia-smi

# Install and upgrade dependencies
!pip install --upgrade ultralytics roboflow pyyaml pillow opencv-python-headless matplotlib seaborn pandas

import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: CUDA not detected! Switch to GPU runtime via Runtime -> Change runtime type -> T4 GPU.")

## 2. Secure Roboflow Authentication (No Hardcoded Secrets)
Provide your free Roboflow API key. You can either:
1. Add a secret named `ROBOFLOW_API_KEY` in the Google Colab Secrets pane (key icon on left toolbar), OR
2. Enter it securely via the interactive prompt below.

*The key is never printed or committed.*

In [ ]:
import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print("Successfully loaded ROBOFLOW_API_KEY from Colab Secrets.")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass.getpass("Enter your Roboflow API key (input hidden): ")

assert ROBOFLOW_API_KEY and len(ROBOFLOW_API_KEY.strip()) > 5, "Invalid API Key provided!"

## 3. Automated Dataset Acquisition
Downloads the three verified public agricultural pest object detection datasets:
1. `teppybugz/recognition-for-pest` (v1) [Aphids, Whitefly, Thrips - 671 images]
2. `ishtar-slt5o/rice-pest-detection-p0ptq` (v1) [Brown Planthopper, Generic Stem Borer - 4,629 images]
3. `asphyxiasea/helicoverpa-armigera-l5tar` (v4) [Helicoverpa armigera - 202 images]

In [ ]:
import os
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
raw_data_dir = "/content/raw_datasets"
os.makedirs(raw_data_dir, exist_ok=True)

# 1. Aphids, Whiteflies, Thrips (TeppyBugz: Object Detection, 671 images)
print("Downloading Dataset 1: teppybugz/recognition-for-pest (v1)...")
proj1 = rf.workspace("teppybugz").project("recognition-for-pest")
ds1 = proj1.version(1).download("yolov8", location=os.path.join(raw_data_dir, "teppybugz_pest_recognition"))

# 2. Paddy Rice Pests (Ishtar: Object Detection, 4,629 images)
print("\nDownloading Dataset 2: ishtar-slt5o/rice-pest-detection-p0ptq (v1)...")
proj2 = rf.workspace("ishtar-slt5o").project("rice-pest-detection-p0ptq")
ds2 = proj2.version(1).download("yolov8", location=os.path.join(raw_data_dir, "ishtar_rice_pests"))

# 3. Helicoverpa armigera (asphyxiaSea: Object Detection, 202 images, Version 4)
print("\nDownloading Dataset 3: asphyxiasea/helicoverpa-armigera-l5tar (v4)...")
proj3 = rf.workspace("asphyxiasea").project("helicoverpa-armigera-l5tar")
ds3 = proj3.version(4).download("yolov8", location=os.path.join(raw_data_dir, "asphyxia_helicoverpa"))

print("\nAll 3 source datasets successfully downloaded.")

## 4. Multi-Source Pipeline: Unification, Remapping, Deduplication & Anti-Leakage Partitioning
- Class remapping to `0..5` schema.
- Prunes exact duplicates via SHA-256.
- Groups Roboflow augmented copies by base parent hash (`extract_parent_group_id`) to prevent data leakage.
- Partitions strictly at the parent group level (70% Train, 20% Val, 10% Test, `seed=42`).

In [ ]:
import re
import yaml
import shutil
import random
import hashlib
from collections import defaultdict
from PIL import Image

UNIFIED_CLASSES = {
    0: "Aphids",
    1: "Whitefly",
    2: "Brown Planthopper",
    3: "Yellow Stem Borer",
    4: "Thrips",
    5: "Helicoverpa armigera"
}

def compute_file_sha256(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

def extract_parent_group_id(filename):
    base = os.path.splitext(filename)[0]
    cleaned = re.sub(r'(_(jpg|jpeg|png))?\.rf\.[a-z0-9]+$', '', base, flags=re.IGNORECASE)
    cleaned = re.sub(r'_(rot\d+|flip|bright\d+|crop\d+)$', '', cleaned, flags=re.IGNORECASE)
    return cleaned.strip()

def verify_image(image_path):
    if not os.path.exists(image_path) or os.path.getsize(image_path) == 0:
        return False, None
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            w, h = img.size
            if w <= 0 or h <= 0:
                return False, None
        return True, (w, h)
    except Exception:
        return False, None

def parse_and_remap_annotation(label_path, class_mapping):
    if not os.path.exists(label_path):
        return True, []
    remapped_boxes = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                src_cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                continue
            if src_cls not in class_mapping:
                continue
            target_cls = class_mapping[src_cls]
            if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                continue
            remapped_boxes.append((target_cls, xc, yc, w, h))
    return True, remapped_boxes

output_dir = "/content/cropguard_dataset"
seen_sha256 = {}
samples = []

# Define Source Ingestion Configurations with Source-Specific Class Mappings
# Dataset 1: 0: Aphids, 1: Whitefly, 2: Thrips (maps to 0, 1, 4)
# Dataset 2: Brown Planthopper -> 2, Stem Borer -> 3 (Green Leafhopper, Rice Bug filtered out)
# Dataset 3: Helicoverpa armigera (0 -> 5)

def build_source_class_mapping(source_names):
    mapping = {}
    for idx, raw_name in enumerate(source_names):
        name = raw_name.lower().strip()
        if 'healthy' in name:
            continue
        if 'aphid' in name:
            mapping[idx] = 0
        elif 'whitefl' in name:
            mapping[idx] = 1
        elif 'brown' in name and ('planthopper' in name or 'hopper' in name):
            mapping[idx] = 2
        elif 'yellow' in name and ('stem' in name and 'borer' in name):
            mapping[idx] = 3
        elif 'stem' in name and 'borer' in name:
            print(f"[BOTANICAL GATE] Class {idx} '{raw_name}' is generic 'stem-borer' (NOT verified as Yellow Stem Borer). Unmapped.")
            continue
        elif 'thrip' in name:
            mapping[idx] = 4
        elif 'helicoverpa' in name or 'armigera' in name or 'bollworm' in name:
            mapping[idx] = 5
    return mapping

def read_source_yaml_names(ds_dir):
    yaml_file = os.path.join(ds_dir, "data.yaml")
    if os.path.exists(yaml_file):
        with open(yaml_file) as f:
            data = yaml.safe_load(f)
            return data.get('names', [])
    return []

# Ingest Dataset 1 (TeppyBugz: HEALTHY, WHITEFLY, APHIDS, THRIPS)
ds1_dir = os.path.join(raw_data_dir, "teppybugz_pest_recognition")
ds1_map = build_source_class_mapping(read_source_yaml_names(ds1_dir))
if not ds1_map: ds1_map = {1: 1, 2: 0, 3: 4}

# Ingest Dataset 2 (Ishtar: brown-planthopper -> 2; generic stem-borer unmapped)
ds2_dir = os.path.join(raw_data_dir, "ishtar_rice_pests")
ds2_map = build_source_class_mapping(read_source_yaml_names(ds2_dir))

# Ingest Dataset 3 (Helicoverpa: helicoverpa-armigera -> 5)
ds3_dir = os.path.join(raw_data_dir, "asphyxia_helicoverpa")
ds3_map = build_source_class_mapping(read_source_yaml_names(ds3_dir))
if not ds3_map: ds3_map = {0: 5}

sources = [
    ("Aphids_Whiteflies_Thrips", ds1_dir, ds1_map),
    ("Paddy_Rice_Pests", ds2_dir, ds2_map),
    ("Helicoverpa_armigera", ds3_dir, ds3_map)
]

excluded_images_count = 0
for sname, sroot, smap in sources:
    print(f"\nProcessing source: {sname} with mapping: {smap}")
    for split in ["train", "valid", "test", ""]: 
        split_imgs = os.path.join(sroot, split, "images") if split else os.path.join(sroot, "images")
        split_lbls = os.path.join(sroot, split, "labels") if split else os.path.join(sroot, "labels")
        if not os.path.exists(split_imgs): continue
        for fname in os.listdir(split_imgs):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in {".jpg", ".jpeg", ".png", ".webp"}: continue
            ipath = os.path.join(split_imgs, fname)
            bname = os.path.splitext(fname)[0]
            lpath = os.path.join(split_lbls, f"{bname}.txt")
            
            ok, dims = verify_image(ipath)
            if not ok: continue
            
            sha = compute_file_sha256(ipath)
            if sha in seen_sha256: continue
            seen_sha256[sha] = ipath
            
            _, boxes = parse_and_remap_annotation(lpath, smap)
            if not boxes:
                excluded_images_count += 1
                continue
            
            pgroup = extract_parent_group_id(fname)
            samples.append({
                "source": sname,
                "image_path": ipath,
                "filename": fname,
                "parent_group": pgroup,
                "boxes": boxes
            })

print(f"\nTotal verified non-duplicate images with target pests: {len(samples)}")
print(f"Total excluded images (zero target pest boxes, e.g. HEALTHY): {excluded_images_count}")

## 5. Calculate Actual Pre-Training Class Distribution & Quality Metrics
Evaluates:
1. Actual bounding-box counts per class.
2. Actual image counts per class.
3. Multi-class images (images containing > 1 pest species).
4. Class balance assessment.

In [ ]:
import pandas as pd

box_counts = defaultdict(int)
image_counts = defaultdict(int)
multi_class_count = 0

for s in samples:
    c_in_s = set()
    for b in s["boxes"]:
        cid = b[0]
        box_counts[cid] += 1
        c_in_s.add(cid)
    for cid in c_in_s:
        image_counts[cid] += 1
    if len(c_in_s) > 1:
        multi_class_count += 1

dist_data = []
for cid in range(6):
    cname = UNIFIED_CLASSES[cid]
    dist_data.append({
        "Class ID": cid,
        "Class Name": cname,
        "Bounding Boxes": box_counts[cid],
        "Images with Class": image_counts[cid]
    })

df_dist = pd.DataFrame(dist_data)
print("=" * 65)
print("ACTUAL CROP黠GUARD AI PRE-TRAINING CLASS DISTRIBUTION")
print("=" * 65)
print(df_dist.to_string(index=False))
print("=" * 65)
print(f"Total Bounding Boxes: {sum(box_counts.values())}")
print(f"Multi-Class Images:   {multi_class_count}")
print(f"Excluded Images:      {excluded_images_count} (zero target pest boxes, e.g. HEALTHY crops)")
print(f"Available Classes:    {[UNIFIED_CLASSES[c] for c in range(6) if box_counts[c] > 0]}")
print(f"Missing Classes:      {[UNIFIED_CLASSES[c] for c in range(6) if box_counts[c] == 0]}")
print("=" * 65)

## 6. Group-Aware Partitioning (70% Train, 20% Val, 10% Test)
Splits images strictly by parent group ID using `seed=42`. Guarantees zero leakage of augmented variants between training and evaluation.

In [ ]:
random.seed(42)
group_to_samples = defaultdict(list)
for s in samples:
    group_to_samples[s["parent_group"]].append(s)

groups = list(group_to_samples.keys())
random.shuffle(groups)

n_total = len(groups)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.20)

train_groups = set(groups[:n_train])
val_groups = set(groups[n_train:n_train + n_val])
test_groups = set(groups[n_train + n_val:])

for sub in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, "images", sub), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", sub), exist_ok=True)

def write_split(group_set, subname):
    count = 0
    for g in group_set:
        for s in group_to_samples[g]:
            fname = s["filename"]
            bname = os.path.splitext(fname)[0]
            dst_img = os.path.join(output_dir, "images", subname, fname)
            dst_lbl = os.path.join(output_dir, "labels", subname, f"{bname}.txt")
            shutil.copy2(s["image_path"], dst_img)
            with open(dst_lbl, "w", encoding="utf-8") as f:
                for b in s["boxes"]:
                    f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")
            count += 1
    return count

n_tr = write_split(train_groups, "train")
n_va = write_split(val_groups, "val")
n_te = write_split(test_groups, "test")

print(f"Splitting Complete:\n- Train: {n_tr} images\n- Val:   {n_va} images\n- Test:  {n_te} images")

# Generate YOLO dataset configuration
dataset_yaml_path = "/content/cropguard_data.yaml"
yaml_config = {
    "path": output_dir,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 6,
    "names": UNIFIED_CLASSES
}

with open(dataset_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

print(f"\nDataset YAML saved to {dataset_yaml_path}")

## 7. Strict Pre-Training Verification Audit
Verifies:
- Zero filename overlap between splits.
- Zero parent group leakage.
- Every annotation within normalized `[0, 1]`.
- All 6 classes represented in the training set.

In [ ]:
train_imgs = set(os.listdir(f"{output_dir}/images/train"))
val_imgs = set(os.listdir(f"{output_dir}/images/val"))
test_imgs = set(os.listdir(f"{output_dir}/images/test"))

assert len(train_imgs.intersection(val_imgs)) == 0, "Train/Val overlap!"
assert len(train_imgs.intersection(test_imgs)) == 0, "Train/Test overlap!"
assert len(val_imgs.intersection(test_imgs)) == 0, "Val/Test overlap!"

tr_parents = {extract_parent_group_id(f) for f in train_imgs}
va_parents = {extract_parent_group_id(f) for f in val_imgs}
te_parents = {extract_parent_group_id(f) for f in test_imgs}

assert len(tr_parents.intersection(va_parents)) == 0, "Parent leakage: Train/Val!"
assert len(tr_parents.intersection(te_parents)) == 0, "Parent leakage: Train/Test!"
assert len(va_parents.intersection(te_parents)) == 0, "Parent leakage: Val/Test!"

# Check class coverage in training split
tr_classes = set()
for lf in os.listdir(f"{output_dir}/labels/train"):
    with open(f"{output_dir}/labels/train/{lf}") as f:
        for l in f:
            p = l.strip().split()
            if p:
                try:
                    tr_classes.add(int(p[0]))
                except ValueError:
                    pass

print("Classes present in training split:", {c: UNIFIED_CLASSES[c] for c in sorted(tr_classes)})
missing_classes = set(range(6)) - tr_classes

# HARD VALIDATION GATE: Strictly prevents training if any class is missing
if missing_classes:
    six_class_gate_passed = False
    missing_names = [f"Class {c} ({UNIFIED_CLASSES[c]})" for c in sorted(missing_classes)]
    gate_msg = (
        f"[TRAINING GATE BLOCKED] Cannot train 6-class CropGuard AI model!\n"
        f"Missing classes in training split: {missing_names}.\n"
    )
    if 3 in missing_classes:
        gate_msg += (
            "\nCRITICAL BLOCKER: Class 3 ('Yellow Stem Borer') has ZERO verified annotations.\n"
            "The downloaded rice dataset contains generic 'stem-borer', which cannot be\n"
            "scientifically mapped to Scirpophaga incertulas without biological evidence.\n"
            "Training is halted until a verified Yellow Stem Borer annotation source is obtained."
        )
    print("\n" + "=" * 72)
    print(gate_msg)
    print("=" * 72)
    raise RuntimeError(gate_msg)
else:
    six_class_gate_passed = True
    print("\n[PASSED] All integrity, anti-leakage, and 6-class representation checks passed!")

## 8. Fine-Tune YOLO11s with Automatic CUDA OOM Fallback
- Model: `yolo11s.pt`
- Resolution: `640x640`
- Batch Size: `16` (Automatic fallback to `8` if CUDA OOM occurs)
- Epochs: `75` (Early stopping `patience=15`)
- Mosaic (`1.0`) & MixUp (`0.15`) for field robustness and class balance.

In [ ]:
from ultralytics import YOLO

# Guard against running training if validation gate was not passed
if not globals().get('six_class_gate_passed', False):
    raise RuntimeError('[TRAINING ABORTED] Pre-training validation gate was not passed! Training cannot proceed with missing classes.')

batch_size = 16
model = YOLO('yolo11s.pt')

try:
    print(f"Attempting YOLO11s fine-tuning with batch={batch_size} on Tesla T4...")
    results = model.train(
        data=dataset_yaml_path,
        epochs=75,
        batch=batch_size,
        imgsz=640,
        patience=15,
        device=0,
        seed=42,
        optimizer='auto',
        lr0=0.01,
        lrf=0.01,
        mosaic=1.0,
        mixup=0.15,
        project='cropguard_yolo',
        name='yolo11s_run',
        save=True,
        save_period=10,
        exist_ok=True,
        verbose=True
    )
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("\nCUDA Out of Memory caught! Clearing cache and falling back to batch=8...")
        torch.cuda.empty_cache()
        batch_size = 8
        model = YOLO('yolo11s.pt')
        results = model.train(
            data=dataset_yaml_path,
            epochs=75,
            batch=batch_size,
            imgsz=640,
            patience=15,
            device=0,
            seed=42,
            optimizer='auto',
            lr0=0.01,
            lrf=0.01,
            mosaic=1.0,
            mixup=0.15,
            project='cropguard_yolo',
            name='yolo11s_run',
            save=True,
            save_period=10,
            exist_ok=True,
            verbose=True
        )
    else:
        raise e

## 9. Independent Test Evaluation & Metrics Export
Evaluates the final checkpoint on the held-out test split for:
- Precision, Recall, F1
- mAP@50 and mAP@50:95
- Class-wise breakdown

In [ ]:
best_weights = '/content/cropguard_yolo/yolo11s_run/weights/best.pt'
trained_model = YOLO(best_weights)

# Run independent evaluation on the held-out test split
metrics = trained_model.val(data=dataset_yaml_path, split='test', device=0)

print("\n" + "=" * 55)
print("FINAL HELD-OUT TEST EVALUATION METRICS")
print("=" * 55)
print(f"mAP @ 0.50     : {metrics.box.map50 * 100:.2f}%")
print(f"mAP @ 0.50:0.95: {metrics.box.map * 100:.2f}%")
print(f"Mean Precision : {metrics.box.mp * 100:.2f}%")
print(f"Mean Recall    : {metrics.box.mr * 100:.2f}%")
print("=" * 55)

## 10. Display Diagnostic Curves & Confusion Matrix

In [ ]:
from IPython.display import Image as IPImage, display

diagnostic_plots = [
    '/content/cropguard_yolo/yolo11s_run/confusion_matrix.png',
    '/content/cropguard_yolo/yolo11s_run/confusion_matrix_normalized.png',
    '/content/cropguard_yolo/yolo11s_run/results.png',
    '/content/cropguard_yolo/yolo11s_run/PR_curve.png',
    '/content/cropguard_yolo/yolo11s_run/F1_curve.png'
]

for p in diagnostic_plots:
    if os.path.exists(p):
        print(f"\nDiagnostic Plot: {os.path.basename(p)}")
        display(IPImage(p))

## 11. Export Artifacts & Backup to Google Drive
Saves:
1. `best.pt` & `last.pt`
2. `best.onnx`
3. `results.csv`
4. All diagnostic evaluation images
Directly to Google Drive for permanent persistence and local deployment.

In [ ]:
# 1. Export to ONNX
trained_model.export(format='onnx', dynamic=False, simplify=True)

# 2. Save directly to Google Drive
from google.colab import drive
drive.mount('/content/drive')

drive_dest = '/content/drive/MyDrive/CropGuard_AI'
os.makedirs(f"{drive_dest}/weights", exist_ok=True)
os.makedirs(f"{drive_dest}/evaluation", exist_ok=True)

# Copy model checkpoints
shutil.copy2('/content/cropguard_yolo/yolo11s_run/weights/best.pt', f"{drive_dest}/weights/best.pt")
if os.path.exists('/content/cropguard_yolo/yolo11s_run/weights/last.pt'):
    shutil.copy2('/content/cropguard_yolo/yolo11s_run/weights/last.pt', f"{drive_dest}/weights/last.pt")
if os.path.exists('/content/cropguard_yolo/yolo11s_run/weights/best.onnx'):
    shutil.copy2('/content/cropguard_yolo/yolo11s_run/weights/best.onnx', f"{drive_dest}/weights/best.onnx")

# Copy results.csv and diagnostic plots
if os.path.exists('/content/cropguard_yolo/yolo11s_run/results.csv'):
    shutil.copy2('/content/cropguard_yolo/yolo11s_run/results.csv', f"{drive_dest}/evaluation/results.csv")

for p in diagnostic_plots:
    if os.path.exists(p):
        shutil.copy2(p, f"{drive_dest}/evaluation/{os.path.basename(p)}")

print(f"\nAll trained checkpoints and evaluation metrics successfully saved to Google Drive: {drive_dest}")